In [53]:
import pandas as pd

In [54]:
df = pd.read_parquet("hf://datasets/sharjeelyunus/github-issues-dataset/github_issues_dataset.parquet")

In [55]:
df.drop(["id", "repo"], axis=1, inplace=True)

## Normalize labels column

In [56]:
import re

def clean_labels(raw_labels):
    noise_labels = {"p1", "p2", "p3", "p4", "p5"}
    parts = [l.strip().lower() for l in str(raw_labels).split(",")]
    meaningful = [
        l for l in parts
        if re.search(r"[a-z]", l) and l not in noise_labels
    ]
    return meaningful if meaningful else ["unknown"]

df["labels-list"] = df["labels"].apply(clean_labels)


In [57]:
from collections import Counter

all_labels = [l for row in df["labels-list"] for l in row]
counts = Counter(all_labels)

In [58]:
label_map = {
    "Bug": [
        "bug",
        "c-bug",
        "issue-bug",
        "type: bug",
        "bug-report",
        "kind/bug",
        "crash",
        "c: crash",
        "needsfix",
        "regression",
        "c: regression",
        "freeze-slow-crash-leak",
        "i-crash",
        "bug 🐛",
        "🤖:bug",
        "bug :beetle:",
        "s-bug-has-test",
        "i-ice",
        "bug-vim",
    ],
    "Feature Request": [
        "feature-request",
        "enhancement",
        "suggestion",
        "c: new feature",
        "c: proposal",
        "feature",
        "idea-enhancement",
        "feature request",
        "c-enhancement",
        "c-feature-request",
        "featurerequest",
        "type: feature request",
        "idea-new powertoy",
        "new feature",
        "function request",
        "feat",
        "💡 feature request",
        "kind/feature",
        "type:feature",
        "issue-feature",
        "improvement",
        "experience enhancement",
    ],
    "Performance": [
        "performance",
        "c: performance",
        "module: performance",
        "perf",
        "perf: speed",
        "perf: memory",
        "i-slow",
        "i-heavy",
        "i-compiletime",
        "freeze-slow-crash-leak",
        "module: memory usage",
        "usability",
    ],
    "Documentation": [
        "documentation",
        "docs",
        "doc",
        "module: docs",
        "d: api docs",
        "a-docs",
        "category: documentation",
        "area: docs",
        "addon: docs",
        "d: examples",
        "examples",
    ],
    "Support": [
        "question",
        "question / support",
        "question (invalid tracker)",
        "discussion",
        "asking-for-help-with-local-system-issues",
        "in discussion",
        "under-discussion",
        "site-support-request",
        "request",
    ],
    "Platform-specific": [
        "platform-android",
        "platform-ios",
        "platform-web",
        "platform-windows",
        "platform-linux",
        "platform-mac",
        "platform:windows",
        "platform:macos",
        "platform:android",
        "platform:ios",
        "platform:web",
        "platform:linuxbsd",
        "windows",
        "macos",
        "linux",
        "mobile",
        "os-windows",
        "o-windows",
    ],
    "Needs Triage": [
        "needs-triage",
        "needs triage",
        "triaged",
        "needsinvestigation",
        "needs investigation",
        "awaiting more feedback",
        "needs more info",
        "s-needs-repro",
        "needs-repro",
        "needs reproduction",
        "status: unconfirmed",
        "unconfirmed",
        "info-needed",
        "confirmation-pending",
        "in triage",
        "triage",
    ],
    "Refactoring": [
        "c: tech-debt",
        "c-cleanup",
        "c-optimization",
        "debt",
        "better-engineering",
        "polish",
        "build",
        "testing",
        "a: tests",
        "a-testsuite",
        "module: flaky-tests",
    ],
    "Security": [
        "nsfw",
        "geo-restricted",
        "geo-blocked",
    ],
    "Meta / Process (discard)": [
        "good first issue",
        "help wanted",
        "stale",
        "inactive",
        "wip",
        "rescheduled",
        "skipped",
        "auto-transferred",
        "has workaround",
        "workaround available",
        "patch-available",
        "fix available",
        "newer patch available",
        "waiting for 👍",
        "needs-team-response",
        "team",
        "oncall: pt2",
        "oncall: distributed",
        "oncall: jit",
        "oncall: export",
        "lifecycle/frozen",
        "stat:awaiting tensorflower",
        "asyncawait-triaged",
    ],
}

In [59]:
counts.most_common(500)

[('bug', 20511),
 ('triaged', 12292),
 ('t-compiler', 5930),
 ('needsinvestigation', 5103),
 ('framework', 5019),
 ('needs-triage', 4255),
 ('enhancement', 4075),
 ('c-bug', 3985),
 ('help wanted', 3752),
 ('has reproducible steps', 3605),
 ('topic:editor', 3496),
 ('suggestion', 3289),
 ('feature-request', 3055),
 ('c: new feature', 2945),
 ('c: proposal', 2842),
 ('issue-bug', 2635),
 ('engine', 2626),
 ('team-framework', 2265),
 ('triaged-framework', 2247),
 ('package', 2215),
 ('confirmed', 2175),
 ('a-diagnostics', 2102),
 ('feature', 2010),
 ('tool', 1901),
 ('needs testing', 1898),
 ('topic:rendering', 1893),
 ('needs triage', 1832),
 ('compiler/runtime', 1791),
 ('team-design', 1777),
 ('discussion', 1770),
 ('triaged-design', 1767),
 ('f: material design', 1751),
 ('idea-enhancement', 1665),
 ('feature request', 1650),
 ('documentation', 1608),
 ('team-engine', 1568),
 ('triaged-engine', 1539),
 ('oncall: pt2', 1531),
 ('c-enhancement', 1507),
 ('topic:3d', 1494),
 ('platform-

In [60]:
top_labels = {label for label, _ in counts.most_common(500)}


In [61]:
def reduced_labels(labels):
    return [l if l in top_labels else "other" for l in labels]

df["labels-reduced"] = df["labels-list"].apply(reduced_labels)


In [62]:
alias_to_group = {}
for group, aliases in label_map.items():
    for alias in aliases:
        alias_to_group[alias.lower()] = group

discard_categories = {"Meta / Process (discard)"}

def map_labels_with_dictionary(labels):
    mapped = []
    seen = set()
    for label in labels:
        group = alias_to_group.get(label.lower(), "other")
        if group in discard_categories:
            continue
        if group not in seen:
            mapped.append(group)
            seen.add(group)
    return mapped if mapped else ["unknown"]

df["labels-mapped"] = df["labels-list"].apply(map_labels_with_dictionary)
df["labels-mapped"].explode().value_counts().head(20)


labels-mapped
other                91217
Bug                  33309
Needs Triage         27490
Feature Request      25570
Platform-specific     7876
Support               4944
Documentation         4280
Performance           3806
Refactoring           3192
unknown               1475
Security               370
Name: count, dtype: int64

In [63]:
target_share = 0.83
total = sum(counts.values())
target = target_share * total

running = 0
n = 0
for _, c in counts.most_common():
    running += c
    n += 1
    if running >= target:
        break

print("n =", n)
print("coverage =", running / total)

n = 528
coverage = 0.8300270540492855
